# HouseCrafter on Colab via Python 3.10 sidecar

Kernel stays **3.13**. All HouseCrafter commands use:
`/content/micromamba/envs/housecrafter/bin/python`

Do **not** run Medium `update-alternatives`. GPU **A100** recommended.
Needs official `ckpts/` + `dataRelease/` — not a Pinterest PNG.

## Step 1 — Kernel

In [ ]:
import sys
print("kernel", sys.version.split()[0], sys.executable)
!nvidia-smi -L
!df -h /content | tail -1

## Step 2 — Sidecar 3.10 + Open3D
Skip if you already have `open3d 0.18.0 OK`.

In [ ]:
%cd /content
from pathlib import Path
if not Path("/content/houseCrafter/scripts/colab_py310.sh").exists():
    !git clone --branch feat/gradio-colab-ui --single-branch https://github.com/sourman-dev/houseCrafter.git || git clone https://github.com/sourman-dev/houseCrafter.git
%cd /content/houseCrafter
!git pull --ff-only origin feat/gradio-colab-ui || true
!bash scripts/colab_py310.sh

## Step 3 — Confirm sidecar

In [ ]:
import sys
from pathlib import Path
PY = Path("/content/micromamba/envs/housecrafter/bin/python")
print("kernel still", sys.version.split()[0])
assert PY.exists(), "sidecar missing — rerun Step 2"
!{PY} -c "import sys,open3d; print('sidecar', sys.version.split()[0]); print('open3d', open3d.__version__)"

## Step 4 — Torch 2.1 + HouseCrafter packages into sidecar

In [ ]:
%cd /content/houseCrafter
!git pull --ff-only origin feat/gradio-colab-ui || true
!bash scripts/colab_py310_hc_deps.sh

## Step 5 — Download sample data + checkpoints
Uses sidecar `gdown --folder` (no `--remaining-ok`). Quota errors come from Google Drive, not this flag.

In [ ]:
%cd /content/houseCrafter
!git pull --ff-only origin feat/gradio-colab-ui || true
!bash scripts/colab_download_assets.sh
!echo "---- ckpts ----" && ls -lh ckpts | head -20
!echo "---- dataRelease ----" && ls -lh dataRelease | head -20

## Step 6 — Generate 1 scene

In [ ]:
%cd /content/houseCrafter/src
PY = "/content/micromamba/envs/housecrafter/bin/python"
!{PY} generate_scene.py \
  --data_root ../dataRelease \
  --ckpt_path ../ckpts/3dfront_layout_iodepth_1871_scene_3m \
  --out_dir ../gen_rgbd \
  --start 0 --end 1

## Step 7 — Fuse RGB-D to `.ply`

In [ ]:
%cd /content/houseCrafter/recon_utils
PY = "/content/micromamba/envs/housecrafter/bin/python"
!{PY} get_gen_data.py --base_dir ../gen_rgbd --dst_path ../generated_data_v0 --get_all_scenes True
!{PY} fuse_gen_data.py --gen_dir ../generated_data_v0
!find ../generated_data_v0 -name '*.ply' | head